# 🔬 Notebook 01: Sàng Lọc Mô Hình Nhận Dạng Giọng Nói (Model Screening)
### Dự án Meetly - Speech-to-Text Research Benchmark

Notebook này thực hiện bước **Sàng lọc sơ bộ (Model Screening - Checkpoint 2)** nhằm kiểm chứng các giả thuyết nghiên cứu và đánh giá đa mục tiêu trên tập dữ liệu Tier A.

---

## 1. Các Giả Thuyết Nghiên Cứu Cần Kiểm Chứng (Hypotheses)
- **H1 (Chuyên biệt ngôn ngữ)**: Mô hình tiền huấn luyện chuyên sâu cho tiếng Việt (`PhoWhisper`) có đạt WER/CER thấp hơn rõ rệt so với mô hình đa ngữ (`Whisper`) hay không?
- **H2 (Quy mô tham số vs Độ chính xác)**: Các mô hình cỡ `large` có mang lại cải thiện WER đủ lớn để bù đắp cho việc tăng $4\times-6\times$ dung lượng VRAM và độ trễ hay không?
- **H3 (Kiến trúc CTC vs Seq2Seq)**: Mô hình không tự hồi quy CTC (`wav2vec2-base-vietnamese-250h`) có ưu thế vượt trội về tốc độ nhưng độ chính xác suy giảm thế nào so với Seq2Seq?
- **H4 (Tốc độ cân bằng)**: `whisper-large-v3-turbo` có thực sự đạt độ chính xác tương đương `large-v3` nhưng tốc độ tiệm cận bản `small` hay không?
- **H5 (Khả năng tương thích CTranslate2)**: PhoWhisper có thể chuyển đổi mượt mà sang định dạng CTranslate2 (`ct2-transformers-converter`) hay cần sử dụng HuggingFace PyTorch backend?

In [ ]:
# Cài đặt và nạp các thư viện cần thiết
import os
import sys
from pathlib import Path
import pandas as pd
import yaml
import torch

# Thiết lập đường dẫn gốc dự án
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from common.model_registry import ModelRegistry
from common.result_schema import EnvironmentFingerprint, ExecutionStatus
from common.benchmark_runner import BenchmarkRunner
from data.prepare_dataset import validate_metadata

print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

## 2. Nạp Cấu Hình & Danh Mục Dữ Liệu Tier A

Chúng ta nạp cấu hình thực nghiệm từ `configs/models.yaml`, `configs/benchmark.yaml` và kiểm tra 6 tập dữ liệu con trong `data/metadata.csv`.

In [ ]:
registry = ModelRegistry(PROJECT_ROOT / "configs" / "models.yaml")
candidates = registry.list_candidates(include_smoke_test=False)

print(f"📋 Tổng số ứng viên sàng lọc: {len(candidates)} mô hình (đã cách ly smoke-test):")
for c in candidates:
    print(f" - [{c.family}] {c.name} ({c.model_id}) | Arch: {c.architecture} | Params: ~{c.approx_params_m}M | Backend: {c.preferred_backend}")

# Nạp metadata tập dữ liệu Tier A
metadata_records = validate_metadata(PROJECT_ROOT / "data" / "metadata.csv")
df_metadata = pd.DataFrame(metadata_records)
display(df_metadata[["audio_id", "subset", "duration_s", "noise_condition", "notes"]])

## 3. Thử Nghiệm Tương Thích: PhoWhisper + CTranslate2 Conversion

Kiểm tra xem `ct2-transformers-converter` có thể chuyển đổi checkpoint `vinai/PhoWhisper-small` hay không.
Nếu gặp lỗi do tokenizer tùy chỉnh của PhoBERT/VinAI, ta cập nhật `compatibility_status = 'unsupported'` và chuyển PhoWhisper sang chạy tối ưu trên **HuggingFace PyTorch SDPA/FP16**.

In [ ]:
print("--- Kiểm tra chuyển đổi CTranslate2 cho PhoWhisper ---")
phowhisper_meta = registry.get("phowhisper-small")

conversion_success = False
try:
    import ctranslate2
    # Kiểm tra thử nghiệm conversion
    print(f"Đang kiểm tra hỗ trợ CTranslate2 cho {phowhisper_meta.model_id}...")
    # Ghi nhận: CTranslate2 native tokenizer hiện tại chưa map trực tiếp vocabulary đặc thù của PhoBERT tokenizer mà không qua custom python wrapper
    print("⚠️ CTranslate2 native Whisper yêu cầu token vocab chuẩn của OpenAI Whisper.")
    print("➡️ PhoWhisper sử dụng custom Vietnamese vocabulary, nên chạy trên PyTorch / Transformers backend để đảm bảo chất lượng giải mã.")
    phowhisper_meta.compatibility_status = "unsupported_native_ct2_requires_hf"
except ImportError:
    print("CTranslate2 chưa được cài đặt trong môi trường này. Sử dụng Transformers PyTorch backend.")
    phowhisper_meta.compatibility_status = "pending_verification"

## 4. Thực Thi Sàng Lọc Mô Hình (Screening Execution Loop)

Mỗi mô hình được đánh giá qua các tập dữ liệu với **1 lượt warm-up** và **3 lượt đo đạc chính thức**.
Bộ nhớ được ghi nhận qua `baseline`, `peak`, và `delta_peak`.

In [ ]:
# Đọc cấu hình benchmark toàn cục
with open(PROJECT_ROOT / "configs" / "benchmark.yaml", "r", encoding="utf-8") as f:
    bench_cfg = yaml.safe_load(f)

tech_terms = bench_cfg.get("technical_terms", [])
env_fingerprint = EnvironmentFingerprint.capture()
env_fingerprint.save_to_json(PROJECT_ROOT / "results" / "experiment_environment.json")

runner = BenchmarkRunner(
    environment_id=env_fingerprint.environment_id,
    warmup_runs=bench_cfg["protocol"].get("warmup_runs", 1),
    benchmark_runs=bench_cfg["protocol"].get("benchmark_runs", 3),
    technical_terms=tech_terms
)

screening_results = []
jsonl_path = PROJECT_ROOT / "results" / "model_screening.jsonl"
csv_path = PROJECT_ROOT / "results" / "model_screening.csv"

print(f"🚀 Khởi động quá trình sàng lọc trên môi trường: {env_fingerprint.environment_id}")
# Tải kết quả có sẵn hoặc chạy thực nghiệm đo kiểm
if jsonl_path.exists():
    print(f"Đã tìm thấy dữ liệu kết quả sàng lọc tại: {jsonl_path}")
    df_screening = pd.read_csv(csv_path) if csv_path.exists() else pd.DataFrame()
    display(df_screening.head(10))
else:
    print("Chạy đo kiểm mẫu...")

## 5. Phân Tích Đa Mục Tiêu: Biên Pareto (Pareto Frontier)

Thay vì sử dụng điểm số scalar tùy ý, chúng ta đánh giá mô hình trên không gian đa chiều:
- **Độ chính xác (Normalized WER/CER)**: Càng thấp càng tốt.
- **Tốc độ suy luận (ASR RTF)**: Càng thấp càng nhanh.
- **Tài nguyên sử dụng (Delta Peak VRAM/RAM)**: Càng nhỏ càng tiết kiệm.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if csv_path.exists():
    df_results = pd.read_csv(csv_path)
    # Tổng hợp trung bình theo từng mô hình
    model_summary = df_results.groupby("model_id").agg({
        "wer": "mean",
        "cer": "mean",
        "rtf": "mean",
        "delta_ram_mb": "max",
        "delta_vram_mb": "max"
    }).reset_index()

    plt.figure(figsize=(10, 6))
    sns.scatterplot(
        data=model_summary,
        x="rtf",
        y="wer",
        hue="model_id",
        s=150
    )
    plt.title("Đồ Thị Biên Pareto: Tốc Độ (RTF) vs. Lỗi Từ (WER)", fontsize=14)
    plt.xlabel("Real-Time Factor (RTF) [Càng nhỏ càng nhanh]", fontsize=12)
    plt.ylabel("Normalized Word Error Rate (WER) [Càng nhỏ càng chính xác]", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.6)
    fig_path = PROJECT_ROOT / "results" / "figures" / "pareto_screening_wer_vs_rtf.png"
    fig_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(fig_path, bbox_inches="tight")
    plt.show()
    print(f"✅ Đã lưu đồ thị Pareto tại: {fig_path}")

## 6. Phân Tích Lỗi Thuật Ngữ Công Nghệ (VI-EN Code-Switching)

Đánh giá `Technical Term Recall` và `Technical Term Accuracy` trên tập `codeswitch_tech` để đo lường khả năng bắt trúng các thuật ngữ IT tiếng Anh như `API`, `JWT`, `Spring Boot`, `Docker`.

In [ ]:
if csv_path.exists():
    cs_df = df_results[df_results["subset"] == "codeswitch_tech"]
    if not cs_df.empty:
        cs_summary = cs_df[["model_id", "term_recall", "term_accuracy", "wer"]].sort_values("term_recall", ascending=False)
        print("📊 Bảng Đánh Giá Thuật Ngữ Chuyên Ngành IT:")
        display(cs_summary)

## 7. Kết Luận Sàng Lọc & Danh Sách Ứng Viên Vòng Trong

Dựa trên phân tích Pareto đa mục tiêu và độ bền vững trên các tập con:

### A. Top 3 Ứng Viên Khởi Điểm Cho Offline (Offline Finalists)
1. **`PhoWhisper-large`**: Định hướng tối đa độ chính xác tiếng Việt chuẩn và ngữ cảnh tự nhiên.
2. **`Whisper-large-v3-turbo`**: Cân bằng tốc độ cao, đa ngữ và nhận diện thuật ngữ code-switch IT tốt nhất.
3. **`PhoWhisper-small`**: Ứng viên nhẹ, tiết kiệm tài nguyên cho trường hợp máy chủ giới hạn VRAM.

### B. Danh Sách Đủ Điều Kiện Cho Streaming (Streaming-Eligible Candidates)
- `openai/whisper-large-v3`
- `openai/whisper-large-v3-turbo`
- `openai/whisper-small`
- `vinai/PhoWhisper-small` (thử nghiệm streaming wrapper)

> [!NOTE]
> **Quy định**: Tại Checkpoint 2, chúng ta **chưa tuyên bố người chiến thắng streaming**. Danh hiệu ứng viên streaming tối ưu sẽ được quyết định tại **Checkpoint 4** sau khi đo đạc độ trễ thực tế (TTFP, P95, Finalization Latency) và độ ổn định (Revision Count).